<a href="https://colab.research.google.com/github/cc-huang-0716/internship_tests/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install'] + 'pdfplumber'.split(), check=True)
import pdfplumber

# 讀取並合併兩個法律文檔
def extract_text_from_pdf(file_path):
    texts = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            page_texts = page.extract_text()
            if page_texts:
                texts += page_texts + "\n"
    return texts

text1 = extract_text_from_pdf("/content/text.pdf")
text2 = extract_text_from_pdf("/content/text1.pdf")

combined_text = text1 + "\n" + "_"*40 + "\n" + text2
#print(combined_text) # <- 查看文檔用

In [ ]:
import numpy as np
import pandas as pd

def Optimization():
    key_word = [ "第", "條", "項", "款", "本法", "本條", "下列各款", "依規定", "依前條", "依本法",
    "依主管機關規定", "依第", "得", "應", "不得", "但書", "除外", "以上", "以下", "之規定",
    "所稱", "所定", "所為", "所需", "本章", "本節", "章節", "規定如左", "本細則",
    "準用", "準此辦理", "公告", "報經", "查核", "核准", "核備", "備查", "解釋令","主管機關", "金融監督管理委員會", "金管會", "金融機構", "銀行業", "信託業", "證券商",
    "期貨商", "保險業", "金融控股公司", "信用合作社", "農會信用部", "資產管理公司",
    "存款保險", "資本適足率", "流動性比率", "內部控制", "內部稽核", "風險控管", "法遵",
    "KYC", "AML", "洗錢防制", "恐怖分子資金防制", "資訊安全", "財報揭露", "重大訊息公告",
    "公開說明書", "公開發行公司", "興櫃公司", "證券交易法", "票券金融", "債券市場",
    "資產證券化", "信用評等", "合格投資人", "自然人", "法人", "保戶權益", "財務重組",
    "不良債權", "利率上限", "準備金", "流動準備", "超額準備", "放款總額", "風險性資產",   "公司章程", "股東會", "董事會", "董事", "獨立董事", "監察人", "監事會", "經理人",
    "發起人", "公司登記", "商業登記", "營業登記", "營利事業", "法人", "自然人", "分公司",
    "子公司", "出資額", "股份總數", "普通股", "特別股", "盈餘分配", "資本公積", "保留盈餘",
    "財報", "合併", "分割", "解散", "清算", "破產", "債權人", "票據", "商業帳簿", "會計帳冊",
    "商業會計法", "營業秘密", "競業禁止", "職務發明", "技術移轉", "股權轉讓", "股東名冊",
    "無記名股票", "公開收購", "特別決議", "普通決議", "代表訴訟", "法定代表人", "名義負責人", "行政處分", "行政罰", "刑責", "民事責任", "損害賠償", "罰鍰", "違約金", "法院", "主管機關",
    "法律效果", "救濟程序", "異議", "申請", "核發", "審查", "審議", "程序正義",
    "契約", "要約", "承諾", "契約成立", "信賴保護", "誠信原則", "明示", "默示",
    "時效", "起算日", "除權判決", "行政訴訟", "再審", "抗告", "上訴"]

    rw_match = []

    def overlapping_chunking(text,chunk_size,prefix_size):
    # string = []
    # string.append(j for j in text.strip("/n").strip("").strip(".").strip("\n\n").strip(" "))
        chunks = []
        i = 0
        chunk = combined_text[0:chunk_size]
        chunks.append(chunk)

        for i in range(0, len(combined_text), chunk_size):
            chunk = combined_text[i:i+chunk_size]

            ## prefix = chunks[-1][-prefix_size:]
            ## prev = chunk
            ## new_chunk = prefix + prev
            ## chunks.append(new_chunk)
            if chunks:  # 不是第一段
                prefix = chunks[-1][-prefix_size:]  # 從上一段尾巴取 prefix
                chunk = prefix + chunk
                chunks.append(chunk)

        # key_word = [ "第*條"  ]

        keyword_contained_list = []

        ## for chunk in chunks:
        ##   for kw in key_word:
        ##     if kw in chunk:
        ##       keyword_contained_list.append(chunk)

        for chunk in chunks:
            if any(kw in chunk for kw in key_word):
                keyword_contained_list.append(chunk)

        # return len(keyword_contained_list)/ (len(chunks) * len(key_word))
        prob = len(keyword_contained_list)/ len(chunks)
        return prob


    def reward_function(prob,chunk_size, prefix_size):
        rw = 0.5 * prob - 0.01 * chunk_size - 0.005 * prefix_size
        return rw


    rw_match = []

    for i in range(1, len(combined_text)):
        for j in range(1, 10):
            prob = overlapping_chunking(text=combined_text,chunk_size=i,prefix_size=j)
            rw = reward_function(prob=prob, chunk_size=i, prefix_size=j)
            rw_match.append({'chunk_size':i, 'prefix_size':j, 'score':rw})

    return rw_match

df = pd.DataFrame(Optimization())

In [ ]:
splited_text = combined_text.split("\n")
#print(splited_text)   <- 查看文檔用

key_word = [ "第", "條", "項", "款", "本法", "本條", "下列各款", "依規定", "依前條", "依本法",
"依主管機關規定", "依第", "得", "應", "不得", "但書", "除外", "以上", "以下", "之規定",
"所稱", "所定", "所為", "所需", "本章", "本節", "章節", "規定如左", "本細則",
"準用", "準此辦理", "公告", "報經", "查核", "核准", "核備", "備查", "解釋令","主管機關", "金融監督管理委員會", "金管會", "金融機構", "銀行業", "信託業", "證券商",
"期貨商", "保險業", "金融控股公司", "信用合作社", "農會信用部", "資產管理公司",
"存款保險", "資本適足率", "流動性比率", "內部控制", "內部稽核", "風險控管", "法遵",
"KYC", "AML", "洗錢防制", "恐怖分子資金防制", "資訊安全", "財報揭露", "重大訊息公告",
"公開說明書", "公開發行公司", "興櫃公司", "證券交易法", "票券金融", "債券市場",
"資產證券化", "信用評等", "合格投資人", "自然人", "法人", "保戶權益", "財務重組",
"不良債權", "利率上限", "準備金", "流動準備", "超額準備", "放款總額", "風險性資產",   "公司章程", "股東會", "董事會", "董事", "獨立董事", "監察人", "監事會", "經理人",
"發起人", "公司登記", "商業登記", "營業登記", "營利事業", "法人", "自然人", "分公司",
"子公司", "出資額", "股份總數", "普通股", "特別股", "盈餘分配", "資本公積", "保留盈餘",
"財報", "合併", "分割", "解散", "清算", "破產", "債權人", "票據", "商業帳簿", "會計帳冊",
"商業會計法", "營業秘密", "競業禁止", "職務發明", "技術移轉", "股權轉讓", "股東名冊",
"無記名股票", "公開收購", "特別決議", "普通決議", "代表訴訟", "法定代表人", "名義負責人", "行政處分", "行政罰", "刑責", "民事責任", "損害賠償", "罰鍰", "違約金", "法院", "主管機關",
"法律效果", "救濟程序", "異議", "申請", "核發", "審查", "審議", "程序正義",
"契約", "要約", "承諾", "契約成立", "信賴保護", "誠信原則", "明示", "默示",
"時效", "起算日", "除權判決", "行政訴訟", "再審", "抗告", "上訴"]

# 將分句文本中每一句的關鍵詞以list和句子對照起來
kw_list = []
for sentence in splited_text:
  kw_contained_list = [k for k in set(key_word) if k in sentence] # 用set(key_word)避免重複的關鍵字
  if kw_contained_list:
    kw_list.append((sentence, kw_contained_list))

#print(kw_list)  # -> 觀察文檔用

# 轉成字典
kw_dict = dict()
for sentence, keywords in kw_list:
    kw_dict[sentence] = keywords
#print(kw_dict)# -> 觀察文檔用

import numpy as np
import pandas as pd
from itertools import combinations
from scipy.special import softmax

# 建立關鍵字索引表
key_word_set = list(set(key_word))  # 轉為 list，方便索引
key_word_index = {kw: i for i, kw in enumerate(key_word_set)}
index_to_kw = {i: kw for kw, i in key_word_index.items()}

# 初始化共現矩陣
mat = np.zeros((len(key_word_set), len(key_word_set)), dtype=int)

# 計算共現次數
for _, kw in kw_list:
    indices = [key_word_index[k] for k in kw]
    for i, j in combinations(indices, 2):
        mat[i][j] += 1
        mat[j][i] += 1  # 共現對稱

# 加上對角線（單字自己出現的次數）
for _, kw in kw_list:
    indices = [key_word_index[k] for k in kw]
    for i in indices:
        mat[i][i] += 1

# 轉成 DataFrame 方便閱讀
df = pd.DataFrame(mat, index=key_word_set, columns=key_word_set)

# 套用 softmax（橫向）
sf_matrix = softmax(mat, axis=1)
sf_df = pd.DataFrame(sf_matrix, index=key_word_set, columns=key_word_set)

# 顯示結果
print(" 共現矩陣 (前10x10):")
print(df.iloc[:10, :10])

print(" Softmax 關聯度矩陣 (前10x10):")
print(sf_df.iloc[:10, :10])

# 下載
#df.to_csv("co_occurrence_matrix.csv")
#sf_df.to_csv("softmax_matrix.csv")

import pandas as pd
import networkx as nx

# 上傳 softmax 關聯矩陣
df = pd.read_csv('/content/softmax_matrix.csv', index_col=0)

# 建立無向圖
G = nx.Graph()

# 加入所有節點
for kw in df.columns:
    G.add_node(kw)

# 為每個節點找 Top 5 關聯邊
for i in df.index:
    related = df.loc[i].drop(i)  # 排除自己
    top5 = related.sort_values(ascending=False).head(5)

    for j, weight in top5.items():
        # 為避免重複邊（A→B, B→A），只加入一次
        if not G.has_edge(i, j):
            G.add_edge(i, j, weight=weight)

# 列出某節點的關聯詞
def get_top_related(keyword, top_k=5):
    if keyword not in G:
        return []
    neighbors = G[keyword]
    sorted_neighbors = sorted(neighbors.items(), key=lambda x: x[1]['weight'], reverse=True)
    return [n[0] for n in sorted_neighbors[:top_k]]

# 印出「契約」的 Top 5 相關詞
print(f"契約的關聯詞：{get_top_related('契約')}")

# 製作query
matched_kw = [kw for kw in G if kw in input_query]

if matched_kw:
    all_related = []
    for kw in matched_kw:
        related = get_top_related(kw, top_k=5)
        all_related.extend(related)

    # 去重後加入原提問中
    all_related = list(set(all_related))
    input_q = f"{input_query}，也請一併考慮：{', '.join(all_related)}"
else:
    input_q = input_query

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 熱力圖
pivot = df.pivot(index="chunk_size", columns="prefix_size", values="score")

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Chunking Parameter Reward Heatmap")
plt.xlabel("Prefix Size")
plt.ylabel("Chunk Size")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 3D圖
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.plot_trisurf(df["chunk_size"], df["prefix_size"], df["score"], cmap='viridis', edgecolor='none')

ax.set_xlabel('Chunk Size')
ax.set_ylabel('Prefix Size')
ax.set_zlabel('Score')
ax.set_title('3D Surface of Chunking Reward')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#折線圖
plt.figure(figsize=(12, 6))
for prefix in sorted(df["prefix_size"]):
    subset = df[df["prefix_size"]]
    plt.plot(subset["chunk_size"], subset["score"], marker="o", label=f"Prefix {prefix}")

plt.title("Reward vs Chunk Size for Different Prefix Sizes")
plt.xlabel("Chunk Size")
plt.ylabel("Reward Score")
plt.legend(title="Prefix Size")
plt.grid(True)
plt.tight_layout()
plt.show()